# Clustering / Semantic Search

[sentence-transformers](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)

In [ ]:
!pip install -U sentence-transformers
!pip install sentence-transformers scikit-learn

In [ ]:
from sentence_transformers import SentenceTransformer
sentences = ["How do I reset my password?", "I forgot my password. How can I get back in?"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
#print(embeddings)


In [7]:
from sentence_transformers import util
print("Shape:", embeddings.shape)

Shape: (2, 384)


In [8]:
score = util.cos_sim(embeddings[0], embeddings[1]).item()
print(f"Cosine similarity: {score:.3f}")

Cosine similarity: 0.818


In [9]:
print(util.cos_sim(embeddings, embeddings))

tensor([[1.0000, 0.8179],
        [0.8179, 1.0000]])


In [11]:
from sentence_transformers import SentenceTransformer, util
from sklearn.cluster import KMeans

# 1. Short texts to organize and search
tickets = [
    "I forgot my password and cannot log in.",
    "How can I reset my account password?",
    "My login credentials are not working.",
    "Where is my package?",
    "My order has not arrived yet.",
    "Can I track my delivery?",
    "I want a refund for my purchase.",
    "How do I return a damaged item?",
    "What is your return policy?",
]

# 2. Turn each ticket into a 384-number embedding
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(tickets, normalize_embeddings=True)

print("Embedding shape:", embeddings.shape)  # (9, 384)

# ---- CLUSTERING: discover groups of similar tickets ----
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(embeddings)

print("\nClusters:")
for cluster_id in range(3):
    print(f"\nCluster {cluster_id}:")
    for ticket, assigned_cluster in zip(tickets, cluster_ids):
        if assigned_cluster == cluster_id:
            print(f"  - {ticket}")

# ---- SEMANTIC SEARCH: find tickets related to a new question ----
query = "I can't access my account. How do I change my password?"
query_embedding = model.encode(query, normalize_embeddings=True)

results = util.semantic_search(
    query_embedding,
    embeddings,
    top_k=3
)[0]

print(f"\nSearch query: {query}")
for result in results:
    print(f"{result['score']:.3f}  {tickets[result['corpus_id']]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (9, 384)

Clusters:

Cluster 0:
  - Where is my package?
  - My order has not arrived yet.
  - Can I track my delivery?

Cluster 1:
  - I want a refund for my purchase.
  - How do I return a damaged item?
  - What is your return policy?

Cluster 2:
  - I forgot my password and cannot log in.
  - How can I reset my account password?
  - My login credentials are not working.

Search query: I can't access my account. How do I change my password?
0.840  How can I reset my account password?
0.742  I forgot my password and cannot log in.
0.658  My login credentials are not working.
